# ClinScribe de-ID name/place tagger — LoRA fine-tune (Phase 5)

Runs on **Google Colab or Kaggle (free GPU)**. Synthetic data only (`finetune/make_dataset.py`, Faker, no real people).

1. Runtime → Change runtime type → **T4 GPU**.
2. Run all cells; upload `train.jsonl`, `dev.jsonl`, `test.jsonl` from `data/finetune/deid_ner/` when asked.
3. Download `deid_ner_lora.zip` (adapter + metrics) and put it in `data/finetune/` on the D: drive.

Base model: `distilbert-base-cased` (Apache-2.0). LoRA (PEFT) on the attention query/value projections.

In [ ]:
!pip -q install transformers==4.46.3 peft==0.13.2 datasets==3.1.0 seqeval==1.2.2 accelerate==1.1.1

In [ ]:
import json, os
from google.colab import files  # on Kaggle: add the three files as a dataset and set DATA below
uploaded = files.upload()
DATA = '.'
splits = {s: [json.loads(l) for l in open(os.path.join(DATA, f'{s}.jsonl'))] for s in ('train', 'dev', 'test')}
print({k: len(v) for k, v in splits.items()})

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
LABELS = ['O', 'B-PERSON', 'I-PERSON', 'B-LOCATION', 'I-LOCATION']
L2I = {l: i for i, l in enumerate(LABELS)}
BASE = 'distilbert-base-cased'
tok = AutoTokenizer.from_pretrained(BASE)

def encode(batch):
    enc = tok(batch['tokens'], is_split_into_words=True, truncation=True, max_length=128)
    labels = []
    for i, tags in enumerate(batch['tags']):
        prev, row = None, []
        for w in enc.word_ids(batch_index=i):
            row.append(-100 if w is None or w == prev else L2I[tags[w]])
            prev = w
        labels.append(row)
    enc['labels'] = labels
    return enc

data = {k: Dataset.from_list(v).map(encode, batched=True, remove_columns=['tokens', 'tags']) for k, v in splits.items()}

In [ ]:
import numpy as np
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
from seqeval.metrics import classification_report, f1_score

model = AutoModelForTokenClassification.from_pretrained(BASE, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=L2I)
model = get_peft_model(model, LoraConfig(task_type=TaskType.TOKEN_CLS, r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['q_lin', 'v_lin']))
model.print_trainable_parameters()

def decode(pred):
    logits, labels = pred
    out_p, out_l = [], []
    for p, l in zip(logits.argmax(-1), labels):
        keep = l != -100
        out_p.append([LABELS[i] for i in p[keep]]); out_l.append([LABELS[i] for i in l[keep]])
    return out_p, out_l

def metrics(pred):
    p, l = decode(pred)
    return {'f1': f1_score(l, p)}

args = TrainingArguments('out', learning_rate=5e-4, per_device_train_batch_size=32, num_train_epochs=3,
                         eval_strategy='epoch', save_strategy='no', report_to=[], seed=42)
trainer = Trainer(model=model, args=args, train_dataset=data['train'], eval_dataset=data['dev'],
                  data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
pred = trainer.predict(data['test'])
p, l = decode((pred.predictions, pred.label_ids))
report = classification_report(l, p, digits=3)
print(report)
os.makedirs('deid_ner_lora', exist_ok=True)
model.save_pretrained('deid_ner_lora'); tok.save_pretrained('deid_ner_lora')
open('deid_ner_lora/test_report.txt', 'w').write(report)
!zip -qr deid_ner_lora.zip deid_ner_lora
files.download('deid_ner_lora.zip')